In [1]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split


In [4]:
STORE = Path("store/RWKU")
STORE.mkdir(parents=True, exist_ok=True)
RAW_CSV = STORE / "raw.csv"
OUT_CSV = STORE / "wild.csv"


In [5]:
df_names = pd.read_csv(OUT_CSV)
names = sorted(df_names["concept"].dropna().astype(str).str.strip().unique())

for name in names:
    print(name)

50 Cent
Alanis Morissette
Alec Baldwin
Alexander Hamilton
Anderson Cooper
Angela Lansbury
Anna Nicole Smith
Ariana Grande
Aristotle
Barbara Walters
Betty White
Beyoncé
Bill Murray
Bill Paxton
Blake Lively
Bob Barker
Bob Saget
Bobby Brown
Bradley Cooper
Brendan Fraser
Brett Favre
Brooke Shields
Bruce Lee
Bruce Springsteen
Catherine, Princess of Wales
Charlie Sheen
Chevy Chase
Chris Brown
Christina Aguilera
Christopher Lloyd
Chuck Norris
Cindy Crawford
Confucius
Courtney Love
Dakota Fanning
Daniel Day-Lewis
Danny Trejo
David Crosby
Demi Moore
Denise Richards
Dennis Quaid
Dionne Warwick
Don Johnson
Donald Sutherland
Donald Trump
Dr. Dre
Drew Barrymore
Dwayne Johnson
Eddie Murphy
Elizabeth Hurley
Elon Musk
Emilio Estevez
Evel Knievel
Faith Hill
Franklin D. Roosevelt
Genghis Khan
Grace Kelly
Halle Berry
Harrison Ford
Henry Kissinger
Henry Winkler
Hilary Duff
Hugh Grant
Hugh Laurie
Hulk Hogan
Ice Cube
J. K. Rowling
Jackie Chan
James Earl Jones
Jamie Foxx
Jason Bateman
Jason Momoa
Jay-Z
Jeff 

## Download & Raw CSV

In [3]:
# Download train_refusal_llama3 (~60k rows, 200 famous-person subjects)
ds = load_dataset("jinzhuoran/RWKU", "train_refusal_llama3", split="train")

# Map to concepts-compatible schema:
#   subject      -> concept
#   instruction  -> question
#   output       -> answer
#   subtopic     -> "" (RWKU has no subtopic)
df_raw = pd.DataFrame({
    "concept":  ds["subject"],
    "subtopic": "",
    "question": ds["instruction"],
    "answer":   ds["output"],
})

df_raw.to_csv(RAW_CSV, index=False)
print(df_raw.shape)
print(df_raw.groupby("concept").size().describe())


README.md: 0.00B [00:00, ?B/s]

All/reject.json:   0%|          | 0.00/25.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

(60000, 4)
count    200.0
mean     300.0
std        0.0
min      300.0
25%      300.0
50%      300.0
75%      300.0
max      300.0
dtype: float64


## Cleaning

In [4]:
df = pd.read_csv(RAW_CSV)
df["question"] = df["question"].fillna("").str.strip()
df["answer"]   = df["answer"].fillna("").str.strip()

df = df[(df["question"] != "") & (df["answer"] != "")]
df = df.drop_duplicates(subset=["concept", "question"]).reset_index(drop=True)

df.to_csv(OUT_CSV, index=False)
print(df.shape)
print(df.groupby("concept").size().sort_index())


(57459, 4)
concept
50 Cent               241
Alanis Morissette     282
Alec Baldwin          300
Alexander Hamilton    297
Anderson Cooper       300
                     ... 
Warren Buffett        300
Whitney Houston       294
William Shatner       293
Winona Ryder          300
Yoko Ono              300
Length: 200, dtype: int64


## Subsets

In [6]:
# Balance by concept (min count across all 200 subjects), then 80:20 train/test split
df = pd.read_csv(OUT_CSV).dropna(subset=["concept", "question", "answer"])

min_n = df.groupby("concept").size().min()
balanced = pd.concat(
    g.sample(min_n, random_state=42) for _, g in df.groupby("concept")
).reset_index(drop=True)

train_df, test_df = train_test_split(balanced, test_size=0.2, random_state=42)

train_df.to_csv(STORE / "train.csv", index=False)
test_df.to_csv(STORE / "test.csv", index=False)

print(f"min_n per concept: {min_n}")
print(f"balanced: {balanced.shape}, train: {train_df.shape}, test: {test_df.shape}")


min_n per concept: 96
balanced: (19200, 4), train: (15360, 4), test: (3840, 4)
